# PIRNN Crystallization

This notebook runs a configurable physics-informed RNN.

**Configurable parameters** (edit the cell below):
- `noise_std` — noise level (0.0, 0.1, 0.3, 1.0)
- `solubility_shift` — shift on Ceq (1.0 = none, 1.10 = 10% shift)
- `physics_lambda` — weight on physics loss (e.g. 1e-2, 1, 1e2)
- `data_size_sweep` — list of training set sizes
- `epochs`, `seed`, etc.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import set_seed, get_device
from src.data_generation import generate_runs, generate_synthetic_data
from src.models import PBM, PIRNN
from src.losses import lossX, lossG, lossSmooth
from src.training import train_sweep, evaluate_model
from src.evaluation import reevaluate_params_on_test

device = get_device()
print(f'Using device: {device}')

## Configuration

Edit this cell to change the sweep parameters.

In [ ]:
# ============== SWEEP CONFIGURATION ==============

# Data generation
N_RUNS = 100              # total synthetic runs
NOISE_STD = 0.0           # noise fraction: 0.0, 0.1, 0.3, 1.0
SOLUBILITY_SHIFT = 1.0    # solubility multiplier: 1.0 (none), 1.10 (10% shift)
N_TRAIN = 80              # train/test split
DATA_SEED = 42            # seed for data generation

# Training
DATA_SIZE_SWEEP = [5, 10, 20, 40, 60]
PHYSICS_LAMBDA = 1.0      # weight on physics loss
EPOCHS = 400000
MODEL_SEED = 42            # seed for model initialisation
DROPOUT = 0.2

# Evaluation
EVAL_PHYSICS_IN_VAL = True  # True for noise sweeps, False for solushift

# Output
TAG = 'noiseless'         # tag for saved filenames
OUTPUT_DIR = '../sweep_results'

# =================================================

## Generate Synthetic Data

In [ ]:
set_seed(DATA_SEED)
runs, meta = generate_runs(n=N_RUNS, seed=DATA_SEED)
data = generate_synthetic_data(
    runs, meta,
    noise_std=NOISE_STD,
    solubility_shift=SOLUBILITY_SHIFT,
    n_train=N_TRAIN,
    seed=DATA_SEED,
)

print(f'Train: {data["y_true"].shape[0]} runs, '
      f'Test: {data["y_true_test"].shape[0]} runs, '
      f'Time steps: {data["y_true"].shape[1]}')
print(f'y_scale: {data["y_scale"]}')

### Visualise temperature profiles

In [ ]:
plt.figure(figsize=(6, 3.5))
for label, df in list(runs.items())[:]:
    plt.plot(df['t_min'], df['T_C'], label=label)
plt.xlabel('Time [min]')
plt.ylabel('Temperature [°C]')
plt.title('Sample temperature profiles')
plt.tight_layout()
plt.show()

### Visualise generated data

In [ ]:
state_names = ['$\\mu_0$', '$\\mu_1$', '$\\mu_2$', '$\\mu_3$', 'Concentration']

fig, axes = plt.subplots(2, 3, figsize=(15, 8), tight_layout=True)
axes = axes.flatten()

y = data['y_true'] * data['y_scale']  # un-normalise
for run_idx in range(y.shape[0]):
    for s in range(5):
        axes[s].plot(y[run_idx, :, s], '--')

for s in range(5):
    axes[s].set_title(state_names[s])
    axes[s].set_xlabel('Time [min]')
axes[5].axis('off')
plt.show()

## Run Training Sweep

In [ ]:
results, loss_history, best_val_model = train_sweep(
    data=data,
    device=device,
    data_size_sweep=DATA_SIZE_SWEEP,
    physics_lambda=PHYSICS_LAMBDA,
    epochs=EPOCHS,
    seed=MODEL_SEED,
    eval_physics_in_val=EVAL_PHYSICS_IN_VAL,
    dropout=DROPOUT,
)

## Save Results

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'saved_model'), exist_ok=True)

loss_history_df = pd.DataFrame(loss_history)
loss_history_df.to_csv(f'{OUTPUT_DIR}/loss_history_{TAG}.csv', index=False)

torch.save(best_val_model, f'{OUTPUT_DIR}/saved_model/best_val_{TAG}.pth')

print(f'Saved loss history and models with tag: {TAG}')

## Visualise Training

In [ ]:
loss_history_df = pd.DataFrame(loss_history)

plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 3, figsize=(20, 12), sharey=True)
axes = axes.flatten()

unique_sizes = sorted(loss_history_df['data_size'].unique())
for idx, size in enumerate(unique_sizes):
    if idx >= len(axes):
        break
    subset = loss_history_df[loss_history_df['data_size'] == size]
    ax = axes[idx]
    ax.semilogy(subset['epoch'], subset['train_loss'], label='Train', alpha=0.7)
    ax.semilogy(subset['epoch'], subset['val_loss'], label='Val', alpha=0.7)
    ax.semilogy(subset['epoch'], subset['test_loss'], label='Test', alpha=0.7)
    ax.set_title(f'Data Size = {size}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()

for idx in range(len(unique_sizes), len(axes)):
    axes[idx].axis('off')

plt.suptitle(f'Training curves — {TAG}', fontsize=14)
plt.tight_layout()
plt.show()

## Visualise Predictions (Best Model)

In [ ]:
# Load best model for the largest data size
best_size = str(DATA_SIZE_SWEEP[-1])
ckpt = best_val_model[best_size]

from src.models import PBM, PIRNN

rhs = PBM(y_scale=data['y_scale']).to(device)
model = PIRNN().to(device)
model.load_state_dict(ckpt['model_state'])
rhs.load_state_dict(ckpt['rhs_state'])

# Predict on test set
x0_test = torch.tensor(data['initial_conditions_test'], dtype=torch.float32).to(device)
T_data_test = torch.tensor(data['T_input_test'], dtype=torch.float32).to(device)
y_data_test = torch.tensor(data['y_true_test'], dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    pred_test = model(x0_test, T_data_test)

# Plot one test experiment
exp_no = 4
fig, ax = plt.subplots(2, 3, figsize=(14, 6), tight_layout=True)
for s, (r, c) in enumerate([(0,0), (0,1), (0,2), (1,0), (1,1)]):
    ax[r, c].plot(pred_test[exp_no, :, s].cpu().numpy(), '--', label='PIRNN')
    ax[r, c].plot(y_data_test[exp_no, :, s].cpu().numpy(), label='True')
    ax[r, c].set_title(state_names[s])
    ax[r, c].legend()
ax[1, 2].axis('off')
plt.suptitle(f'Test predictions — {TAG} (data size={best_size})', fontsize=14)
plt.show()

## Re-evaluate Learned Parameters via ODE Integration

In [ ]:
re_eval_results = reevaluate_params_on_test(
    loss_history_df, data,
    eval_epoch_step=5000,
    max_epoch=EPOCHS,
)

re_eval_df = pd.DataFrame(re_eval_results)
re_eval_df.to_csv(f'{OUTPUT_DIR}/re_eval_results_{TAG}.csv', index=False)

# Merge with loss history
combined_df = pd.merge(loss_history_df, re_eval_df, on=['data_size', 'epoch'], how='left')

# Plot re-evaluation MSE
fig, ax = plt.subplots(figsize=(8, 5))
for size in DATA_SIZE_SWEEP:
    sub = combined_df[combined_df['data_size'] == size].dropna(subset=['re_eval_test_mse'])
    ax.semilogy(sub['epoch'], sub['re_eval_test_mse'], label=f'n={size}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Re-eval Test MSE')
ax.set_title(f'ODE re-evaluation — {TAG}')
ax.legend()
plt.tight_layout()
plt.show()

## Parameter Recovery

In [ ]:
from src.constants import GT_PARAMS

param_names = ['kb2', 'alfa', 'beta', 'kg', 'Eag', 'gama_g']
rhs_cols = [f'rhs_{p}' for p in param_names]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), tight_layout=True)
axes = axes.flatten()

for idx, (pname, col) in enumerate(zip(param_names, rhs_cols)):
    ax = axes[idx]
    for size in DATA_SIZE_SWEEP:
        sub = loss_history_df[loss_history_df['data_size'] == size]
        ax.plot(sub['epoch'], np.exp(sub[col]), label=f'n={size}', alpha=0.7)
    ax.axhline(GT_PARAMS[pname], color='k', linestyle='--', label='Ground truth')
    ax.set_title(pname)
    ax.set_xlabel('Epoch')
    ax.set_yscale('log')
    ax.legend(fontsize=7)

plt.suptitle(f'Parameter recovery — {TAG}', fontsize=14)
plt.show()